# Etapa 3 — Análise de Desempenho de Vendas

Notebook executável da Etapa 3. A lógica canônica fica em `etapa3_desempenho_vendas.py`; este notebook reexecuta o script e apresenta os principais arquivos gerados.

## Metodologia resumida

- Fonte única: `data/processed/vendas_tratadas.parquet`.
- Quantidade analisada em `QTD_ARMAZENAGEM`.
- Duas visões explícitas: rede completa e rede física sem Loja 93.
- `TRANSACOES` é contagem de linhas de venda, usada como proxy porque a base não possui id de cupom/pedido/nota.
- Participações, rankings e ABC calculados dentro de cada universo.
- Comparação 2025 vs 2024 feita em anos completos presentes na base.

In [1]:
import runpy
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'outputs' / 'etapa3'
SCRIPT = ROOT / 'notebooks' / 'etapa3_desempenho_vendas.py'

runpy.run_path(str(SCRIPT), run_name='__main__')

Carregando vendas tratadas...



--- Totais por universo ---
REDE_COMPLETA            | Receita: R$   482.5M | Qtd arm.:  4,641,884 | Linhas venda: 1,090,390 | SKUs: 2,729
REDE_FISICA_SEM_LOJA93   | Receita: R$   329.2M | Qtd arm.:  4,544,808 | Linhas venda: 1,065,507 | SKUs: 2,729

Calculando rankings, hierarquias, lojas e sazonalidade...


Validando consistência dos totais...
Salvando arquivos auditáveis...



--- Destaques ---
Loja 93: 31.8% da receita da rede completa
Categorias NIVEL_1 na rede completa: 23
Validações OK: 60/60

[OK] Arquivos salvos em outputs/etapa3/
  outputs\etapa3\curva_abc_produtos.csv
  outputs\etapa3\decomposicao_queda_2025_categorias.csv
  outputs\etapa3\decomposicao_queda_2025_lojas.csv
  outputs\etapa3\desempenho_categorias_n1.csv
  outputs\etapa3\desempenho_categorias_n2.csv
  outputs\etapa3\desempenho_categorias_n3.csv
  outputs\etapa3\desempenho_lojas.csv
  outputs\etapa3\desempenho_regioes.csv
  outputs\etapa3\impacto_loja93.csv
  outputs\etapa3\notas_metodologicas.csv
  outputs\etapa3\ranking_produtos_quantidade.csv
  outputs\etapa3\ranking_produtos_receita.csv
  outputs\etapa3\recomendacoes_melhoria.csv
  outputs\etapa3\resumo_etapa3.md
  outputs\etapa3\sazonalidade_picos_quedas.csv
  outputs\etapa3\validacoes_etapa3.csv
  outputs\etapa3\vendas_mensais.csv


{'__name__': '__main__',
 '__doc__': '\netapa3_desempenho_vendas.py\n===========================\nEtapa 3 — Análise de Desempenho de Vendas\n\nObjetivos\n---------\n1. Rankear produtos por receita e quantidade vendida em unidade de armazenagem\n2. Classificar produtos pela curva ABC de receita\n3. Analisar desempenho por hierarquia de produto, loja e região\n4. Medir sazonalidade mensal e comparar 2024 vs 2025\n5. Decompor a queda de 2025 por categoria e loja\n\nPremissas e decisões metodológicas\n----------------------------------\n- Todos os números partem de data/processed/vendas_tratadas.parquet.\n- Quantidade é analisada em QTD_ARMAZENAGEM, mantendo consistência com as\n  etapas anteriores.\n- A Loja 93 (Alhandra-PB) é mantida na visão de rede completa e segregada na\n  visão de rede física sem Loja 93. A exclusão nunca é implícita.\n- Participações percentuais são sempre calculadas dentro do próprio universo\n  analisado.\n- Variação 2025 vs 2024 compara anos completos presentes 

## Arquivos gerados

In [2]:
arquivos = pd.DataFrame(
    [{'arquivo': str(p.relative_to(ROOT)), 'tamanho_kb': round(p.stat().st_size / 1024, 1)} for p in sorted(OUT.glob('*'))]
)
arquivos

,arquivo,tamanho_kb
0,outputs\etapa3\curva_abc_produtos.csv,1522.5
1,outputs\etapa3\decomposicao_queda_2025_categor...,9.0
2,outputs\etapa3\decomposicao_queda_2025_lojas.csv,4.7
3,outputs\etapa3\desempenho_categorias_n1.csv,11.7
4,outputs\etapa3\desempenho_categorias_n2.csv,61.9
5,outputs\etapa3\desempenho_categorias_n3.csv,225.2
6,outputs\etapa3\desempenho_lojas.csv,6.1
7,outputs\etapa3\desempenho_regioes.csv,4.7
8,outputs\etapa3\impacto_loja93.csv,0.6
9,outputs\etapa3\notas_metodologicas.csv,1.8


## Validações

In [3]:
validacoes = pd.read_csv(OUT / 'validacoes_etapa3.csv')
validacoes.groupby('STATUS').size().reset_index(name='qtde')

,STATUS,qtde
0,OK,60


In [4]:
validacoes.head(10)

,VALIDACAO,OBSERVADO,ESPERADO,DIFERENCA,STATUS
0,Dimensão sem nulos após Etapa 1: DESCRICAO,0.000000e+00,0.000000e+00,0.0,OK
1,Dimensão sem nulos após Etapa 1: NIVEL_1,0.000000e+00,0.000000e+00,0.0,OK
2,Dimensão sem nulos após Etapa 1: NIVEL_2,0.000000e+00,0.000000e+00,0.0,OK
3,Dimensão sem nulos após Etapa 1: NIVEL_3,0.000000e+00,0.000000e+00,0.0,OK
4,Dimensão sem nulos após Etapa 1: CD_CIDADE,0.000000e+00,0.000000e+00,0.0,OK
5,Dimensão sem nulos após Etapa 1: CD_ESTADO,0.000000e+00,0.000000e+00,0.0,OK
6,ranking_receita soma RECEITA - REDE_COMPLETA,4.825157e+08,4.825157e+08,0.0,OK
7,ranking_receita soma QTD_ARMAZENAGEM - REDE_CO...,4.641884e+06,4.641884e+06,0.0,OK
8,ranking_receita soma TRANSACOES - REDE_COMPLETA,1.090390e+06,1.090390e+06,0.0,OK
9,ranking_quantidade soma RECEITA - REDE_COMPLETA,4.825157e+08,4.825157e+08,0.0,OK


## Principais recortes

In [5]:
ranking_receita = pd.read_csv(OUT / 'ranking_produtos_receita.csv')
ranking_receita.query("UNIVERSO == 'REDE_COMPLETA'").head(10)[[
    'RANK_RECEITA', 'CODIGO', 'DESCRICAO', 'NIVEL_1', 'RECEITA',
    'PARTICIPACAO_RECEITA_PCT', 'RECEITA_ACUM_PCT', 'CURVA_ABC_RECEITA'
]]

,RANK_RECEITA,CODIGO,DESCRICAO,NIVEL_1,RECEITA,PARTICIPACAO_RECEITA_PCT,RECEITA_ACUM_PCT,CURVA_ABC_RECEITA
0,1,467774,COND.SPLIT 9000 COND.S3UQ09 INV. 143,D - ELETROS,1.249263e+07,2.589061,2.589061,A
1,2,467770,COND.SPLIT 9000 COND.S3UQ09 INV. 141,D - ELETROS,9.956997e+06,2.063559,4.652620,A
2,3,432048,MASSA CORRIDA PVA CORAL PLS 25KG,S - TINTAS E QUIMICOS,9.903878e+06,2.052550,6.705170,A
3,4,470176,LAV.ROUPA 13.0KG BWK13AB BR M220,D - ELETROS,8.685360e+06,1.800016,8.505186,A
4,5,454103,REFRIG. 2P 386L FF CRM44ABB BR M220,D - ELETROS,7.017098e+06,1.454273,9.959459,A
5,6,467773,COND.SPLIT 9000 EVAP.S3NQ09 INV. 143,D - ELETROS,6.160594e+06,1.276765,11.236225,A
6,7,442061,TV LED 43 FHD SMT 43LM6370PSB,R - ELETRONICOS,6.150607e+06,1.274696,12.510920,A
7,8,457797,COND.JANEL 7000 REM D6NRND1A M220,D - ELETROS,5.447493e+06,1.128977,13.639898,A
8,9,481697,TV LED 55 UHD SMT 4K UN55DU7700GXZ,R - ELETRONICOS,5.359282e+06,1.110696,14.750594,A
9,10,467769,COND.SPLIT 9000 EVAP.S3NQ09 INV. 141,D - ELETROS,5.274701e+06,1.093167,15.843760,A


In [6]:
ranking_receita.query("UNIVERSO == 'REDE_FISICA_SEM_LOJA93'").head(10)[[
    'RANK_RECEITA', 'CODIGO', 'DESCRICAO', 'NIVEL_1', 'RECEITA',
    'PARTICIPACAO_RECEITA_PCT', 'RECEITA_ACUM_PCT', 'CURVA_ABC_RECEITA'
]]

,RANK_RECEITA,CODIGO,DESCRICAO,NIVEL_1,RECEITA,PARTICIPACAO_RECEITA_PCT,RECEITA_ACUM_PCT,CURVA_ABC_RECEITA
2729,1,432048,MASSA CORRIDA PVA CORAL PLS 25KG,S - TINTAS E QUIMICOS,9.903878e+06,3.008151,3.008151,A
2730,2,457797,COND.JANEL 7000 REM D6NRND1A M220,D - ELETROS,5.447493e+06,1.654592,4.662743,A
2731,3,447307,VENT.MES.40CM 3V 2X1 VF42 PT M220,D - ELETROS,4.931448e+06,1.497852,6.160595,A
2732,4,457804,COND.JANEL 10000 REM D6NRND2A M220,D - ELETROS,3.747557e+06,1.138263,7.298858,A
2733,5,487782,TV LED 50 UHD SMT 4K 50UT8050PSA,R - ELETRONICOS,3.520265e+06,1.069226,8.368084,A
2734,6,442061,TV LED 43 FHD SMT 43LM6370PSB,R - ELETRONICOS,3.331930e+06,1.012023,9.380107,A
2735,7,481697,TV LED 55 UHD SMT 4K UN55DU7700GXZ,R - ELETRONICOS,3.229990e+06,0.981060,10.361167,A
2736,8,448186,TV LED 32 HD SMT UN32T4300AGXZD,R - ELETRONICOS,2.907094e+06,0.882985,11.244152,A
2737,9,149992,MARTELETE ROT/ROMP. 800W HR2470 220,F - FERRAMENTAS,2.440676e+06,0.741318,11.985470,A
2738,10,481696,TV LED 50 UHD SMT 4K UN50DU7700GXZ,R - ELETRONICOS,2.261151e+06,0.686790,12.672259,A


In [7]:
categorias = pd.read_csv(OUT / 'desempenho_categorias_n1.csv')
categorias.sort_values(['UNIVERSO', 'RECEITA'], ascending=[True, False]).groupby('UNIVERSO').head(8)[[
    'UNIVERSO', 'NIVEL_1', 'RECEITA', 'PARTICIPACAO_RECEITA_PCT',
    'RECEITA_2024', 'RECEITA_2025', 'VAR_RECEITA_2025_VS_2024_PCT'
]]

,UNIVERSO,NIVEL_1,RECEITA,PARTICIPACAO_RECEITA_PCT,RECEITA_2024,RECEITA_2025,VAR_RECEITA_2025_VS_2024_PCT
0,REDE_COMPLETA,D - ELETROS,1.975705e+08,40.945928,1.450775e+08,5.249304e+07,-63.817246
1,REDE_COMPLETA,C - PISOS E REVESTIMENTOS,5.464366e+07,11.324742,3.703059e+07,1.761307e+07,-52.436427
2,REDE_COMPLETA,R - ELETRONICOS,4.786878e+07,9.920667,2.521489e+07,2.265389e+07,-10.156711
3,REDE_COMPLETA,I - ILUMINACAO,2.532342e+07,5.248206,1.627912e+07,9.044297e+06,-44.442350
4,REDE_COMPLETA,"K - CAMA, MESA E BANHO",2.321228e+07,4.810679,1.733369e+07,5.878588e+06,-66.085774
5,REDE_COMPLETA,O - MOVEIS,2.272289e+07,4.709254,1.713745e+07,5.585440e+06,-67.407993
6,REDE_COMPLETA,B - UTILIDADES DOMESTICAS,2.251381e+07,4.665923,1.556090e+07,6.952916e+06,-55.318031
7,REDE_COMPLETA,U - METAIS E ACESSORIOS,2.036140e+07,4.219841,1.397289e+07,6.388503e+06,-54.279317
23,REDE_FISICA_SEM_LOJA93,D - ELETROS,8.144220e+07,24.736816,5.668311e+07,2.475909e+07,-56.320170
24,REDE_FISICA_SEM_LOJA93,C - PISOS E REVESTIMENTOS,5.464366e+07,16.597173,3.703059e+07,1.761307e+07,-52.436427


In [8]:
lojas = pd.read_csv(OUT / 'desempenho_lojas.csv')
lojas.sort_values(['UNIVERSO', 'RECEITA'], ascending=[True, False])[[
    'UNIVERSO', 'COD_EMPRESA', 'CD_CIDADE', 'CD_ESTADO', 'TIPO_OPERACAO',
    'RECEITA', 'PARTICIPACAO_RECEITA_PCT', 'TRANSACOES', 'TICKET_MEDIO_TRANSACAO'
]].head(15)

,UNIVERSO,COD_EMPRESA,CD_CIDADE,CD_ESTADO,TIPO_OPERACAO,RECEITA,PARTICIPACAO_RECEITA_PCT,TRANSACOES,TICKET_MEDIO_TRANSACAO
0,REDE_COMPLETA,93,ALHANDRA,PB,LOJA_93_ATACADO_B2B,1.532810e+08,31.767045,24883,6160.068825
1,REDE_COMPLETA,3,SALVADOR,BA,REDE_FISICA,6.259999e+07,12.973668,163108,383.794739
2,REDE_COMPLETA,2,RECIFE,PE,REDE_FISICA,5.337709e+07,11.062249,168222,317.301492
3,REDE_COMPLETA,4,RECIFE,PE,REDE_FISICA,4.330823e+07,8.975507,168176,257.517321
4,REDE_COMPLETA,6,JOAO PESSOA,PB,REDE_FISICA,3.481680e+07,7.215682,126733,274.725619
5,REDE_COMPLETA,92,CABO DE SANTO AGOSTINHO,PE,REDE_FISICA,2.856488e+07,5.919990,9107,3136.585558
6,REDE_COMPLETA,5,ARACAJU,SE,REDE_FISICA,2.352973e+07,4.876469,94289,249.549065
7,REDE_COMPLETA,7,NATAL,RN,REDE_FISICA,2.225043e+07,4.611337,93598,237.723319
8,REDE_COMPLETA,8,CARUARU,PE,REDE_FISICA,2.134132e+07,4.422927,81056,263.291050
9,REDE_COMPLETA,9,SALVADOR,BA,REDE_FISICA,2.106606e+07,4.365880,100680,209.237786


In [9]:
mensal = pd.read_csv(OUT / 'vendas_mensais.csv')
mensal[[
    'UNIVERSO', 'ANO_MES', 'RECEITA', 'QTD_ARMAZENAGEM', 'TRANSACOES',
    'RECEITA_YOY_DELTA', 'RECEITA_YOY_VAR_PCT'
]].tail(12)

,UNIVERSO,ANO_MES,RECEITA,QTD_ARMAZENAGEM,TRANSACOES,RECEITA_YOY_DELTA,RECEITA_YOY_VAR_PCT
36,REDE_FISICA_SEM_LOJA93,2025-01,1.426736e+07,204399.72,50770,-3.258518e+06,-18.592610
37,REDE_FISICA_SEM_LOJA93,2025-02,1.254009e+07,189086.35,43073,-2.295144e+06,-15.470904
38,REDE_FISICA_SEM_LOJA93,2025-03,1.211978e+07,157182.93,39728,-4.241049e+06,-25.921966
39,REDE_FISICA_SEM_LOJA93,2025-04,1.267997e+07,158019.41,36761,-4.254152e+06,-25.121772
40,REDE_FISICA_SEM_LOJA93,2025-05,1.334420e+07,192808.44,35454,-5.571453e+06,-29.454188
41,REDE_FISICA_SEM_LOJA93,2025-06,8.324730e+06,125186.78,28238,-8.298506e+06,-49.921121
42,REDE_FISICA_SEM_LOJA93,2025-07,9.222148e+06,127248.08,28720,-7.983580e+06,-46.400710
43,REDE_FISICA_SEM_LOJA93,2025-08,7.130348e+06,109634.68,23467,-9.492882e+06,-57.106121
44,REDE_FISICA_SEM_LOJA93,2025-09,6.236496e+06,95079.35,19378,-9.466554e+06,-60.284811
45,REDE_FISICA_SEM_LOJA93,2025-10,6.420794e+06,81249.47,19243,-1.396254e+07,-68.499788


## Resumo interpretativo

In [10]:
display(Markdown((OUT / 'resumo_etapa3.md').read_text(encoding='utf-8')))

# Etapa 3 — Análise de Desempenho de Vendas

## Escopo executado

- Rankings de produtos por receita e por quantidade em unidade de armazenagem, com participação, acumulado e curva ABC por receita.
- Visões separadas para rede completa e rede física sem Loja 93.
- Agregações por `NIVEL_1`, `NIVEL_2`, `NIVEL_3`, loja, cidade/estado e mês.
- Comparação 2024 vs 2025 e decomposição da queda por categoria e por loja.

## Principais achados

- A rede completa soma R$ 482,5M em receita, 1.090.390 linhas de venda (proxy de transações) e 2.729 SKUs ativos.
- A Loja 93 responde por 31,8% da receita, mas por 2,3% das linhas de venda. A receita média por linha dela é R$ 6.160,07, sinalizando operação fora do padrão da rede física.
- Sem a Loja 93, a rede física soma R$ 329,2M e 1.065.507 linhas de venda.
- Produto líder por receita na rede completa: `467774` — COND.SPLIT 9000 COND.S3UQ09 INV. 143, com R$ 12,5M (2,6% do universo).
- Produto líder por receita na rede física: `432048` — MASSA CORRIDA PVA CORAL PLS     25KG, com R$ 9,9M (3,0% do universo).
- Produto líder por quantidade na rede completa: `429455` — TOM.CJ.1T 2P+T 10A        LGX030 POP, com 281.011 unidades de armazenagem.
- A cauda dos rankings também é auditável: `RANK_RECEITA_MENOR` e `RANK_QUANTIDADE_MENOR` identificam os produtos de menor venda. Na rede completa, a menor receita é do produto `435055` (R$ 321,08) e a menor quantidade é do produto `430480` (1,00 unidade de armazenagem).
- A curva A concentra 80,0% da receita em 522 SKUs na rede completa; sem a Loja 93, concentra 80,0% em 713 SKUs.
- Foram observadas 23 categorias de nível 1. A maior categoria na rede completa é `D - ELETROS`, com R$ 197,6M; na rede física, `D - ELETROS`, com R$ 81,4M.
- A loja de maior receita na rede completa é 93 (ALHANDRA-PB), com R$ 153,3M. Sem a Loja 93, a líder é 3 (SALVADOR-BA), com R$ 62,6M.
- A receita caiu -54,2% em 2025 vs 2024 na rede completa e -47,6% na rede física sem Loja 93.
- O maior mês por receita na rede completa foi 2024-11, com R$ 47,2M; o menor foi 2025-12, com R$ 3,9M.
- A maior contribuição bruta para a queda de receita em 2025, por categoria na rede completa, veio de `D - ELETROS` (R$ -92,6M). Por loja, veio da loja 93 (ALHANDRA-PB), com R$ -76,4M.

## Limitações e cuidados metodológicos

- A análise é descritiva: variações e picos não são atribuídos a preço, demanda, ruptura ou captura de dados sem evidência adicional.
- `TRANSACOES` representa linhas de venda, não cupons únicos. A base processada não possui id de cupom, pedido ou nota. Isto está documentado como limitação relevante e recomendação de melhoria em `notas_metodologicas.csv` e `recomendacoes_melhoria.csv`.
- Ticket médio foi mantido como proxy de receita por linha de venda; preço médio foi calculado como receita por unidade de armazenagem.
- A Loja 93 é operação B2B/atacado e distorce médias, rankings e sazonalidade. Por isso todos os outputs trazem `UNIVERSO`.
- A queda de 2025 é comparada contra 2024 completo, usando apenas datas presentes em `vendas_tratadas.parquet`.
- Recomendações de melhoria foram registradas para pontos que limitam a leitura profissional dos dados: id de transação, canal formal da Loja 93, movimentações de estoque, variáveis causais da queda e dicionário de métricas.

## Validações

- 60 validações executadas, todas com status `OK`.
- Somas de receita, quantidade e linhas de venda dos rankings, categorias, lojas e meses batem com os totais de cada universo.
- As decomposições de queda por categoria e por loja fecham com o delta anual 2025 vs 2024.
- A soma Loja 93 + rede física sem Loja 93 fecha com a rede completa.
- As dimensões críticas (`NIVEL_1`, `NIVEL_2`, `NIVEL_3`, cidade e estado) não possuem nulos na base processada usada na análise.

## Como executar

```bash
cd notebooks
python etapa3_desempenho_vendas.py
```

Os arquivos auditáveis são gravados em `outputs/etapa3/`.
